In [28]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [15]:
from google.colab import files
uploaded = files.upload()

Saving loan_approval_dataset.csv to loan_approval_dataset (1).csv


In [16]:
try:
    df = pd.read_csv('loan_approval_dataset.csv')
    print("Dataset loaded successfully!")
    print(df.head())
except FileNotFoundError:
    print("Error: Please upload 'loan_approval_dataset.csv' to the Colab file sidebar.")

Dataset loaded successfully!
   loan_id   no_of_dependents      education  self_employed   income_annum  \
0        1                  2       Graduate             No        9600000   
1        2                  0   Not Graduate            Yes        4100000   
2        3                  3       Graduate             No        9100000   
3        4                  3       Graduate             No        8200000   
4        5                  5   Not Graduate            Yes        9800000   

    loan_amount   loan_term   cibil_score   residential_assets_value  \
0      29900000          12           778                    2400000   
1      12200000           8           417                    2700000   
2      29700000          20           506                    7100000   
3      30700000           8           467                   18200000   
4      24200000          20           382                   12400000   

    commercial_assets_value   luxury_assets_value   bank_asset_value 

In [17]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop the unique identifier 'loan_id'
df = df.drop('loan_id', axis=1)

print("Cleaned Columns:", df.columns.tolist())

Cleaned Columns: ['no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


In [20]:
le = LabelEncoder()

# Encoding features and target
df['education'] = le.fit_transform(df['education'].str.strip())
df['self_employed'] = le.fit_transform(df['self_employed'].str.strip())
df['loan_status'] = le.fit_transform(df['loan_status'].str.strip())

print("Data after encoding:")
print(df[['education', 'self_employed', 'loan_status']].head())

Data after encoding:
   education  self_employed  loan_status
0          0              0            0
1          1              1            1
2          0              0            1
3          0              0            1
4          1              1            1


In [21]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (3415, 11)
Testing set size: (854, 11)


In [22]:
scaler = StandardScaler()

# Fit only on the training data to avoid data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

print("KNN Model training complete.")

KNN Model training complete.


In [26]:
y_pred = knn_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 89.23%

Confusion Matrix:
[[483  53]
 [ 39 279]]

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.90      0.91       536
           1       0.84      0.88      0.86       318

    accuracy                           0.89       854
   macro avg       0.88      0.89      0.89       854
weighted avg       0.89      0.89      0.89       854



In [29]:
# Save to disk
pickle.dump(knn_model, open("loan_knn_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("Files 'loan_knn_model.pkl' and 'scaler.pkl' are ready for download.")

Files 'loan_knn_model.pkl' and 'scaler.pkl' are ready for download.


In [30]:
def predict_new_applicant(data_list):
    """
    Input order: [no_of_dependents, education, self_employed, income_annum,
                  loan_amount, loan_term, cibil_score, residential_assets,
                  commercial_assets, luxury_assets, bank_assets]
    """
    # Scale input
    data_scaled = scaler.transform([data_list])
    # Predict
    result = knn_model.predict(data_scaled)
    return "Approved" if result[0] == 0 else "Rejected"

# Test Example (using values from a typical approved profile)
test_data = [2, 0, 0, 9600000, 29900000, 12, 778, 2400000, 17600000, 22700000, 8000000]
print(f"The predicted status for the test data is: {predict_new_applicant(test_data)}")

The predicted status for the test data is: Approved


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
